# Full-data DistilBERT + word hybrid ensemble

Second submission candidate. Previous fold-0 submission: **0.316 public score**, reported by the user.

Attach the **same private llm-fingerprint-fold0 dataset** and **Name That LLM competition** as before. Select **GPU T4 x2**, turn **Internet Off**, then **Run All**. No new input upload is needed.

This notebook trains both components on all official labeled training rows. It starts fine-tuning from the original frozen encoder, not the previously fine-tuned checkpoint. Nine epochs and equal probability weights were fixed from prior grouped validation. This run has no held-out validation score and is not guaranteed to improve the leaderboard score.

Saving with Save & Run All and submitting will rerun training. The completed run writes submission.parquet and a reusable model ZIP. If you manually rerun after creating full_data_ensemble, the training cell refuses to overwrite it; restart a clean session to repeat training.


In [ ]:
import os
import sys
import zipfile
from pathlib import Path

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
inputs = Path('/kaggle/input')
scripts = list(inputs.rglob('kaggle_inference.py'))
if not scripts:
    archives = list(inputs.rglob('llm-fingerprint-fold0.zip'))
    if len(archives) != 1:
        raise RuntimeError('Attach the existing llm-fingerprint-fold0 dataset with Add Input.')
    unpacked = Path('/kaggle/working/original_model')
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(unpacked)
    scripts = list(unpacked.rglob('kaggle_inference.py'))
if len(scripts) != 1:
    raise RuntimeError('Attach exactly one copy of our original model dataset.')
original = scripts[0].parent
train_files = [p for p in inputs.rglob('train.csv') if 'name-that-llm' in str(p)]
if len(train_files) != 1:
    raise RuntimeError('Attach the Name That LLM competition using Add Input.')
train_path = train_files[0]
test_path = train_path.with_name('test.csv')
assert test_path.is_file()
print('Original encoder:', original / 'frozen_encoder')
print('Official training data:', train_path)
print('Competition test data:', test_path)


In [ ]:
"""Refit the fixed DistilBERT/hybrid ensemble on all official labeled rows.

No validation or test data is used for selection. Nine fine-tuning epochs were
selected before this run from the existing fold-0 and fold-1 best epochs.
"""
import argparse
import gc
import hashlib
import json
import math
import random
from importlib.metadata import version
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import normalize
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoModel, AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup)


class Passages(Dataset):
    def __init__(self, frame, tokenizer, labels):
        self.texts = frame.text.tolist()
        self.targets = frame.model.map({label: i for i, label in enumerate(labels)}).tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        item = self.tokenizer(self.texts[index], truncation=True, max_length=512)
        item['labels'] = self.targets[index]
        return item


def train_full(train_path, encoder_path, output, epochs=9, device_name='cuda'):
    output, train_path, encoder_path = Path(output), Path(train_path), Path(encoder_path)
    if output.exists() and any(output.iterdir()):
        raise ValueError('Choose a new or empty output directory; existing models are preserved.')
    if epochs < 1:
        raise ValueError('epochs must be positive')
    frame = pd.read_csv(train_path, dtype=str, keep_default_na=False)
    required = ['id', 'prompt_id', 'text', 'model']
    if not set(required).issubset(frame.columns) or frame.empty:
        raise ValueError('Expected labeled competition train.csv')
    if frame.id.duplicated().any() or any(frame[c].str.strip().eq('').any() for c in required):
        raise ValueError('Duplicate IDs or empty training values')
    labels = sorted(frame.model.unique())
    if len(labels) != 18:
        raise ValueError('Expected all 18 model labels')
    if device_name == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('Select GPU T4 x2 in Kaggle, or run in a GPU allocation.')
    device = torch.device(device_name)
    use_amp = device.type == 'cuda'
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    if use_amp:
        torch.cuda.manual_seed_all(42)
    torch.set_num_threads(4)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    output.mkdir(parents=True, exist_ok=True)
    config = dict(train_rows=len(frame), train_prompts=int(frame.prompt_id.nunique()),
                  train_sha256=hashlib.sha256(train_path.read_bytes()).hexdigest(),
                  labels=labels, epochs=epochs, max_length=512, learning_rate=2e-5,
                  batch_size=8, gradient_accumulation=4, seed=42, first_weight=0.5,
                  selection='Fixed epochs from prior grouped validation; no validation on full-data refit',
                  status='training', device=str(device),
                  packages={name: version(name) for name in
                            ['torch', 'transformers', 'numpy', 'pandas', 'scikit-learn', 'joblib', 'scipy']})
    (output / 'training_config.json').write_text(json.dumps(config, indent=2))
    tokenizer = AutoTokenizer.from_pretrained(encoder_path, local_files_only=True)
    model = AutoModel.from_pretrained(encoder_path, local_files_only=True).to(device).eval()
    vectors = []
    print(f'Encoding all {len(frame)} labeled passages', flush=True)
    with torch.inference_mode():
        for start in range(0, len(frame), 16):
            batch = tokenizer(frame.text.iloc[start:start+16].tolist(), padding=True,
                              truncation=True, max_length=512, return_tensors='pt')
            batch = {key: value.to(device) for key, value in batch.items()}
            hidden = model(**batch).last_hidden_state
            mask = batch['attention_mask'].unsqueeze(-1).to(hidden.dtype)
            vectors.append(((hidden * mask).sum(1) / mask.sum(1).clamp(min=1)).cpu())
    model.save_pretrained(output / 'frozen_encoder')
    tokenizer.save_pretrained(output / 'frozen_encoder')
    del model
    gc.collect()
    if use_amp:
        torch.cuda.empty_cache()
    vectors = torch.cat(vectors)
    mean = vectors.mean(0)
    scale = vectors.std(0, unbiased=False).clamp(min=1e-6)
    embeddings = normalize(((vectors - mean) / scale).numpy())
    vectorizer = FeatureUnion([('word', TfidfVectorizer(
        ngram_range=(1, 2), min_df=2, sublinear_tf=True,
        max_features=100000, dtype=np.float32))])
    features = hstack([normalize(vectorizer.fit_transform(frame.text)), csr_matrix(embeddings)], format='csr')
    classifier = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=42)
    classifier.fit(features, frame.model)
    joblib.dump(dict(classifier=classifier, tfidf=vectorizer,
                     embedding_mean=mean.numpy(), embedding_scale=scale.numpy(),
                     embedding_weight=1.0, max_length=512, labels=classifier.classes_.tolist(),
                     head='hybrid', pooling='attention-mask mean', text_features='word'),
                output / 'classifier.joblib')
    del vectors, embeddings, features
    print('Full-data hybrid saved. Starting fresh encoder fine-tuning.', flush=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        encoder_path, num_labels=18, id2label=dict(enumerate(labels)),
        label2id={label: i for i, label in enumerate(labels)}, local_files_only=True,
        ignore_mismatched_sizes=True).to(device)
    loader = DataLoader(Passages(frame, tokenizer, labels), batch_size=8, shuffle=True,
                        generator=torch.Generator().manual_seed(42),
                        collate_fn=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8 if use_amp else None))
    decay, no_decay = [], []
    for name, parameter in model.named_parameters():
        (no_decay if name.endswith('bias') or 'LayerNorm.weight' in name else decay).append(parameter)
    optimizer = torch.optim.AdamW([{'params': decay, 'weight_decay': 0.01},
                                   {'params': no_decay, 'weight_decay': 0.0}], lr=2e-5)
    updates = math.ceil(len(loader) / 4) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(updates * 0.1), updates)
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0
        # Normalize the last accumulation group by its actual number of examples.
        window_examples = 0
        for step, batch in enumerate(loader):
            batch = {key: value.to(device) for key, value in batch.items()}
            count = len(batch['labels'])
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = model(**batch).loss
            scaler.scale(loss * count).backward()
            total_loss += loss.detach().item() * count
            window_examples += count
            if (step + 1) % 4 == 0 or step + 1 == len(loader):
                scaler.unscale_(optimizer)
                for parameter in model.parameters():
                    if parameter.grad is not None:
                        parameter.grad.div_(window_examples)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= previous_scale:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                window_examples = 0
        history.append(dict(epoch=epoch, train_loss=total_loss / len(frame)))
        (output / 'history.json').write_text(json.dumps(history, indent=2))
        print(f'Epoch {epoch}/{epochs}: training loss {history[-1]["train_loss"]:.4f}', flush=True)
    model.save_pretrained(output / 'finetuned_model')
    tokenizer.save_pretrained(output / 'finetuned_model')
    config['status'] = 'complete'
    (output / 'training_config.json').write_text(json.dumps(config, indent=2))
    print('Full-data ensemble saved:', output, flush=True)
    del model
    gc.collect()
    if use_amp:
        torch.cuda.empty_cache()
    return output



In [ ]:
"""Offline inference using the existing fold-0 50/50 ensemble."""
from pathlib import Path
import gc
import joblib
import numpy as np
import pandas as pd
import torch
from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import normalize
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer


def submit(root, test_path, output='/kaggle/working/submission.parquet'):
    root = Path(root)
    frame = pd.read_csv(test_path, dtype={'id': str}, keep_default_na=False)
    assert len(frame) and {'id', 'text'}.issubset(frame.columns)
    assert frame.id.is_unique and frame.id.ne('').all()
    texts = frame.text.tolist()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    torch.set_num_threads(4)
    print('Device:', device, '| passages:', len(texts), flush=True)
    bundle = joblib.load(root / 'classifier.joblib')
    labels = bundle['classifier'].classes_.tolist()

    def run(folder, classification=False):
        tokenizer = AutoTokenizer.from_pretrained(root / folder, local_files_only=True)
        cls = AutoModelForSequenceClassification if classification else AutoModel
        model = cls.from_pretrained(root / folder, local_files_only=True).to(device).eval()
        order = None
        if classification:
            model_labels = [model.config.id2label[i] for i in range(model.config.num_labels)]
            assert len(labels) == 18 and set(labels) == set(model_labels)
            order = [model_labels.index(label) for label in labels]
        parts = []
        with torch.inference_mode():
            for start in range(0, len(texts), 8):
                tokens = tokenizer(texts[start:start+8], padding=True, truncation=True,
                                   max_length=512, return_tensors='pt')
                tokens = {k: v.to(device) for k, v in tokens.items()}
                with torch.autocast(device_type=device.type, dtype=torch.float16,
                                    enabled=classification and device.type == 'cuda'):
                    result = model(**tokens)
                if classification:
                    values = result.logits.float().softmax(-1).cpu().numpy()[:, order]
                else:
                    hidden = result.last_hidden_state
                    mask = tokens['attention_mask'].unsqueeze(-1).to(hidden.dtype)
                    values = ((hidden * mask).sum(1) / mask.sum(1).clamp(min=1)).cpu().numpy()
                parts.append(values)
                if start % 256 == 0:
                    print(folder, min(start+8, len(texts)), '/', len(texts), flush=True)
        del model
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        return np.concatenate(parts)

    embeddings = run('frozen_encoder')
    embeddings = normalize((embeddings - bundle['embedding_mean']) / bundle['embedding_scale'])
    features = hstack([normalize(bundle['tfidf'].transform(texts)),
                       bundle['embedding_weight'] * csr_matrix(embeddings)], format='csr')
    hybrid = bundle['classifier'].predict_proba(features)
    fine = run('finetuned_model', classification=True)
    probabilities = 0.5 * hybrid + 0.5 * fine
    submission = pd.DataFrame({'id': frame.id, 'model': np.asarray(labels)[probabilities.argmax(1)]})
    assert len(submission) == len(frame) and submission.model.isin(labels).all()
    submission.to_parquet(output, index=False)
    print('Saved', output, '| rows:', len(submission))
    return submission


In [ ]:
INFERENCE_SOURCE = '"""Offline inference using the existing fold-0 50/50 ensemble."""\nfrom pathlib import Path\nimport gc\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom scipy.sparse import csr_matrix, hstack\nfrom sklearn.preprocessing import normalize\nfrom transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer\n\n\ndef submit(root, test_path, output=\'/kaggle/working/submission.parquet\'):\n    root = Path(root)\n    frame = pd.read_csv(test_path, dtype={\'id\': str}, keep_default_na=False)\n    assert len(frame) and {\'id\', \'text\'}.issubset(frame.columns)\n    assert frame.id.is_unique and frame.id.ne(\'\').all()\n    texts = frame.text.tolist()\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    torch.set_num_threads(4)\n    print(\'Device:\', device, \'| passages:\', len(texts), flush=True)\n    bundle = joblib.load(root / \'classifier.joblib\')\n    labels = bundle[\'classifier\'].classes_.tolist()\n\n    def run(folder, classification=False):\n        tokenizer = AutoTokenizer.from_pretrained(root / folder, local_files_only=True)\n        cls = AutoModelForSequenceClassification if classification else AutoModel\n        model = cls.from_pretrained(root / folder, local_files_only=True).to(device).eval()\n        order = None\n        if classification:\n            model_labels = [model.config.id2label[i] for i in range(model.config.num_labels)]\n            assert len(labels) == 18 and set(labels) == set(model_labels)\n            order = [model_labels.index(label) for label in labels]\n        parts = []\n        with torch.inference_mode():\n            for start in range(0, len(texts), 8):\n                tokens = tokenizer(texts[start:start+8], padding=True, truncation=True,\n                                   max_length=512, return_tensors=\'pt\')\n                tokens = {k: v.to(device) for k, v in tokens.items()}\n                with torch.autocast(device_type=device.type, dtype=torch.float16,\n                                    enabled=classification and device.type == \'cuda\'):\n                    result = model(**tokens)\n                if classification:\n                    values = result.logits.float().softmax(-1).cpu().numpy()[:, order]\n                else:\n                    hidden = result.last_hidden_state\n                    mask = tokens[\'attention_mask\'].unsqueeze(-1).to(hidden.dtype)\n                    values = ((hidden * mask).sum(1) / mask.sum(1).clamp(min=1)).cpu().numpy()\n                parts.append(values)\n                if start % 256 == 0:\n                    print(folder, min(start+8, len(texts)), \'/\', len(texts), flush=True)\n        del model\n        gc.collect()\n        if device.type == \'cuda\':\n            torch.cuda.empty_cache()\n        return np.concatenate(parts)\n\n    embeddings = run(\'frozen_encoder\')\n    embeddings = normalize((embeddings - bundle[\'embedding_mean\']) / bundle[\'embedding_scale\'])\n    features = hstack([normalize(bundle[\'tfidf\'].transform(texts)),\n                       bundle[\'embedding_weight\'] * csr_matrix(embeddings)], format=\'csr\')\n    hybrid = bundle[\'classifier\'].predict_proba(features)\n    fine = run(\'finetuned_model\', classification=True)\n    probabilities = 0.5 * hybrid + 0.5 * fine\n    submission = pd.DataFrame({\'id\': frame.id, \'model\': np.asarray(labels)[probabilities.argmax(1)]})\n    assert len(submission) == len(frame) and submission.model.isin(labels).all()\n    submission.to_parquet(output, index=False)\n    print(\'Saved\', output, \'| rows:\', len(submission))\n    return submission\n'
import json
import zipfile

# This refits on official train.csv; hidden test labels are never available.
full_model = train_full(
    train_path=train_path,
    encoder_path=original / 'frozen_encoder',
    output=Path('/kaggle/working/full_data_ensemble'),
    epochs=9,
    device_name='cuda',
)
submission = submit(full_model, test_path, '/kaggle/working/submission.parquet')
display(submission.head())

# Preserve the new trained model for later inference-only notebooks.
archive_path = Path('/kaggle/working/llm-fingerprint-full-data.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_STORED) as archive:
    for file in sorted(full_model.rglob('*')):
        if file.is_file():
            archive.write(file, str(file.relative_to(full_model)))
    archive.writestr('kaggle_inference.py', INFERENCE_SOURCE)
print('Model archive:', archive_path)
print('Submit submission.parquet to the competition, not the ZIP.')
